In [ ]:
# package import

import psycopg2
import pandas as pd
import geopandas as gpd
from geopandas import GeoSeries, GeoDataFrame

import math
import numpy as np
import os
import time
import gc
import glob as gb

from tqdm import tqdm
from shapely import wkt
from shapely import geometry
from shapely.geometry import Polygon, Point , MultiPoint, LineString, MultiPolygon
from shapely.ops import unary_union

import matplotlib.pyplot as plt

In [ ]:
#setting 
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'

In [ ]:
#read shape file 
jung_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/중구.shp', encoding='utf-8-sig')
dong_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/동구.shp', encoding='utf-8-sig')
seo_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/서구.shp', encoding='utf-8-sig')
youseong_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/유성구.shp', encoding='utf-8-sig')
daedeok_shp=gpd.read_file('C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/LX법정구역경계_시군구_대전광역시/대전광역시 구별 경계파일/대덕구.shp', encoding='utf-8-sig')

In [ ]:
cty_shp = gpd.GeoDataFrame(pd.concat([jung_shp, dong_shp, seo_shp, youseong_shp, daedeok_shp], ignore_index=True))

In [ ]:
ax=cty_shp.plot()
plt.show()

In [ ]:
cty_shp.to_crs(epsg=5179)
cty_shp.crs

In [ ]:
dst = 500
#target_shp=cty_shp[]  #시도 하나만 하도록!
#---------------  making bbox
# 몇개 셀로 나눠야할지 계산
x_cnt = int(np.ceil((cty_shp.unary_union.envelope.bounds[2]-cty_shp.unary_union.envelope.bounds[0])/dst)) 
y_cnt = int(np.ceil((cty_shp.unary_union.envelope.bounds[3]-cty_shp.unary_union.envelope.bounds[1])/dst))
print(x_cnt,y_cnt)

In [ ]:
#기준 지점 그리기 
orign = [cty_shp.unary_union.envelope.bounds[0], cty_shp.unary_union.envelope.bounds[1]]

# grid cell 만들기
polygons = []
for xi in range(x_cnt):
    for yj in range(y_cnt):
        x1 = orign[0] + (dst * xi)
        x2 = x1 + dst
        y1 = orign[1] + (dst * yj)
        y2 = y1 + dst
        polygons.append(Polygon([(x1,y1), (x1,y2), (x2,y2), (x2,y1)]))

cty_grid = gpd.GeoDataFrame({'geometry': polygons}, crs=cty_shp.crs)

cty_grid = cty_grid[cty_grid.geometry.intersects(cty_shp.unary_union)]

cty_grid.reset_index(drop=True, inplace=True)

In [ ]:
cty_grid.boundary.plot()
plt.show()

In [ ]:
# GeoJSON 파일 경로
geojson_path = "C:/Users/kfq/Downloads/1ST_Final_Project/카셰어링/데이터/대전 데이터/지도/대전광역시 행정동.geojson"

# GeoJSON 파일 읽기
gdf = gpd.read_file(geojson_path)

print(gdf)

# Optional: GeoDataFrame 시각화
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, edgecolor='black', alpha=0.5)
plt.show()

In [ ]:
import geopandas as gpd

# Assuming gdf is your GeoDataFrame

# 새로운 '행정동' 컬럼 생성 (여기서는 'adm_nm' 컬럼에서 '동'을 포함한 부분을 추출하는 예제입니다)
gdf['행정동'] = gdf['adm_nm'].apply(lambda x: x.split(' ')[-1] if '동' in x else 'Unknown')

# 출력해 보기
print(gdf[['adm_nm', '행정동']].head())

In [ ]:
import geopandas as gpd

# Assuming gdf is your GeoDataFrame

# Drop the columns
gdf = gdf.drop(columns=['OBJECTID', 'adm_cd', 'adm_cd2', 'sgg', 'sido'])

# Rename columns
gdf = gdf.rename(columns={
    'adm_nm': '주소',
    'sidonm': '시도',
    'sggnm': '시군구'
})

# Ensure column order
gdf = gdf[['주소', '시도', '시군구', 'geometry', '행정동']]

# Print the updated GeoDataFrame to check
print(gdf.head())

In [ ]:
import pandas as pd
import geopandas as gpd

# Load the CSV file
csv_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\주차장\대전광역시 유무료 주차장 정보 데이터.csv'
parking_df = pd.read_csv(csv_path)

# Group by '행정동' and sum '주차구획수'
grouped_parking = parking_df.groupby('행정동')['주차구획수'].sum().reset_index()
grouped_parking = grouped_parking.rename(columns={'주차구획수': '총주차구획수'})


# Merge the GeoDataFrame with the grouped parking data
merged_gdf = gdf.merge(grouped_parking, on='행정동', how='left')
merged_gdf['총주차구획수'] = merged_gdf['총주차구획수'].fillna(0)

# Save the merged GeoDataFrame to GeoJSON
output_geojson_path = r'C:\Users\kfq\Desktop\merged_gdf.geojson'
merged_gdf.to_file(output_geojson_path, driver='GeoJSON')

# Print the first few rows to verify the result
print(merged_gdf.head())


In [ ]:
# Define the path to the CSV file
csv_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\상권\대전광역시 상권매출.csv'

# Load the CSV file into a DataFrame
sales_df = pd.read_csv(csv_path)

# Define a function to split the address into 시도, 시군구, 행정동
def split_address(address):
    # Split address by space
    parts = address.split()
    
    # Handle cases based on typical address format
    if len(parts) >= 3:
        시도 = parts[0]
        시군구 = parts[1]
        행정동 = ' '.join(parts[2:])  # Join the remaining parts as 행정동
    else:
        시도 = 시군구 = 행정동 = ''
    
    return pd.Series([시도, 시군구, 행정동])

# Apply the function to the 주소 column and create new columns
sales_df[['시도', '시군구', '행정동']] = sales_df['주소'].apply(split_address)

# Display the DataFrame with the new columns
print(sales_df.head())

In [ ]:
import pandas as pd
import re
# Define a function to extract the content within parentheses
def extract_dong(address):
    match = re.search(r'\((.*?)\)', address)
    if match:
        return match.group(1)
    else:
        return ''

# Apply the function to the '주소' column and create/update the '행정동' column
sales_df['행정동'] = sales_df['주소'].apply(extract_dong)

# Display the DataFrame with the updated '행정동' column
print(sales_df[['주소', '행정동']].head())

In [ ]:
# Calculate the sum of 매출액 by 행정동 in sales_df
sales_sum_by_dong = sales_df.groupby('행정동')['매출액'].sum().reset_index()
sales_sum_by_dong.columns = ['행정동', '총매출액']  # Rename the column for clarity

# Merge the sales_sum_by_dong with gdf_merge2 on '행정동'
gdf_merge2 = gdf.merge(sales_sum_by_dong, on='행정동', how='left')

# Fill NaN values in the '총매출액' column with 0
gdf_merge2['총매출액'] = gdf_merge2['총매출액'].fillna(0)

# Save the updated GeoDataFrame to a new GeoJSON file
output_path = r'C:\Users\kfq\Desktop\merged_gdf2.geojson'
gdf_merge2.to_file(output_path, driver='GeoJSON')

# Display the updated GeoDataFrame
print(gdf_merge2.head())


In [ ]:
# Define the path to the CSV file
file_path = r'C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\인구\대전광역시 구별 유동인구 수(2020.12).csv'

# Load the CSV file into a DataFrame
population_df = pd.read_csv(file_path)

# Display the first few rows of the DataFrame to check its structure
print(population_df.head())

In [ ]:
print(gdf_merge2.head())


In [ ]:
population_df.rename(columns={'시군구': '시군구'}, inplace=True)

# Merge the population data with gdf_merge2 on '시군구'
gdf_merge2 = gdf_merge2.merge(population_df, on='시군구', how='left')

# Fill NaN values in the population columns with 0 (if needed)
gdf_merge2['20대'] = gdf_merge2['20대'].fillna(0)
gdf_merge2['30대'] = gdf_merge2['30대'].fillna(0)

# Save the updated GeoDataFrame to a new GeoJSON file
output_path = r'C:\Users\kfq\Desktop\merged_gdf_with_population.geojson'
gdf_merge2.to_file(output_path, driver='GeoJSON')

# Display the updated GeoDataFrame
print(gdf_merge2.head())

In [ ]:
# Save the updated GeoDataFrame to a new GeoJSON file
output_path = r'C:\Users\kfq\Desktop\merged_gdf_with_population.csv'
gdf_merge2.to_csv(output_path, index= False, encoding = "utf-8-sig")


In [ ]:
gdf

In [ ]:
import pandas as pd

# 파일 경로 지정
file_path_1 = r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\교통량\2024년 7월말 자동차 등록현황.csv"
file_path_2 = r"C:\Users\kfq\Downloads\1ST_Final_Project\카셰어링\데이터\대전 데이터\교통량\대전 월별통계(도로별, 차로당).csv"

# CSV 파일 불러오기
car_registration_data = pd.read_csv(file_path_1)
road_traffic_data = pd.read_csv(file_path_2)

# 데이터 확인
print("2024년 7월말 자동차 등록현황 데이터:")
print(car_registration_data.head())

print("\n대전 월별통계(도로별, 차로당) 데이터:")
print(road_traffic_data.head())
